# Lab 02 — Multiclass Attack Classification
AI Cybersecurity Masterclass · Lesson 02 · DSJ THE CRITTERS

## Research question
How well can a baseline model distinguish between attack categories?
AI Cybersecurity Masterclass · Lesson 02 · DSJ THE CRITTERS

The purpose is not merely to obtain one global score. We want to identify which attack classes the model understands well and which classes it confuses.

In [ ]:
from pathlib import Path
import sys, os

REPO_URL = "https://github.com/Jacquelinepersha/ai-cybersecurity-public-labs.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")

# OPTIONAL — GOOGLE COLAB ONLY: clone the repo so src/ actually exists here
try:
    import google.colab
    if not Path(REPO_NAME).exists():
        !git clone {REPO_URL}
    os.chdir(REPO_NAME)
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()

# If running from the notebooks/ folder locally:
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data" / "raw"
RESULTS_DIR = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Data folder :", DATA_DIR)

In [ ]:
# OPTIONAL — GOOGLE COLAB ONLY
# Run this cell if the UNSW-NB15 CSV files are not already in DATA_DIR.

try:
    from google.colab import files

    if not (DATA_DIR / "UNSW_NB15_training-set.csv").exists():
        print(
            "Upload UNSW_NB15_training-set.csv, "
            "UNSW_NB15_testing-set.csv, and optionally UNSW_NB15_features.csv"
        )
        uploaded = files.upload()
        DATA_DIR.mkdir(parents=True, exist_ok=True)

        for filename, content in uploaded.items():
            (DATA_DIR / filename).write_bytes(content)

except ImportError:
    print("Not running in Colab. Put the dataset CSV files in:", DATA_DIR)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

from src.data_loader import load_unsw
from src.preprocessing import split_xy, align_columns, build_preprocessor
from src.features import drop_identifier_like_columns
from src.evaluation import multiclass_report

train, test = load_unsw(DATA_DIR)

X_train, y_train = split_xy(train, "attack_cat")
X_test, y_test = split_xy(test, "attack_cat")
X_train, X_test = align_columns(X_train, X_test)

X_train = drop_identifier_like_columns(X_train)
X_test = drop_identifier_like_columns(X_test)

In [ ]:
counts = y_train.value_counts()
display(counts.rename("training_examples").to_frame())

ax = counts.sort_values().plot(kind="barh", figsize=(8, 5))
ax.set_title("Attack-category distribution")
ax.set_xlabel("Training examples")
plt.show()

In [ ]:
model = Pipeline([
    ("preprocess", build_preprocessor(X_train, scale_numeric=False)),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    )),
])

model.fit(X_train, y_train)
predictions = model.predict(X_test)

report = multiclass_report(y_test, predictions)
display(report.round(3))

In [ ]:
class_rows = report.drop(
    index=["accuracy", "macro avg", "weighted avg"],
    errors="ignore"
)

display(
    class_rows
    .sort_values("f1-score")
    [["precision", "recall", "f1-score", "support"]]
    .head(10)
    .round(3)
)

## Your analysis

- Which attack categories are hardest to detect?
- Do rare classes have lower recall?
- Compare macro F1 and weighted F1.
- Why can a strong weighted score hide a security weakness?